In [1]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.shape)
print(df.head())
print(df.columns)


(7225, 11)
   CGPA  Internships  Projects  Workshops/Certifications  AptitudeTestScore  \
0   7.7            1         1                         0                 69   
1   7.6            2         3                         1                 80   
2   7.0            1         1                         1                 64   
3   7.9            1         3                         2                 90   
4   6.7            0         2                         1                 71   

   SoftSkillsRating ExtracurricularActivities PlacementTraining  SSC_Marks  \
0               4.0                        No                No         55   
1               4.3                       Yes               Yes         57   
2               4.1                        No                No         58   
3               4.8                       Yes               Yes         82   
4               3.7                        No                No         55   

   HSC_Marks PlacementStatus  
0         69  

In [2]:
df = df.drop(columns=[
    "ExtracurricularActivities",
    "SSC_Marks",
    "HSC_Marks",
    "PlacementTraining"
])

print(df.shape)
print(df.columns)


(7225, 7)
Index(['CGPA', 'Internships', 'Projects', 'Workshops/Certifications',
       'AptitudeTestScore', 'SoftSkillsRating', 'PlacementStatus'],
      dtype='object')


In [3]:
df.rename(columns={
    "Internships": "Internship_Experience",
    "Workshops/Certifications": "Certifications",
    "SoftSkillsRating": "Communication_Skills",
    "PlacementStatus": "Placement_Status"
}, inplace=True)


In [4]:
df.columns

Index(['CGPA', 'Internship_Experience', 'Projects', 'Certifications',
       'AptitudeTestScore', 'Communication_Skills', 'Placement_Status'],
      dtype='object')

In [5]:
df["Technical_Skills"] = (
    df["Projects"] * 0.4 +
    df["Certifications"] * 0.3 +
    df["Internship_Experience"] * 0.3
)

df["Technical_Skills"] = (df["Technical_Skills"] / df["Technical_Skills"].max()) * 10


In [6]:
df.head()

,CGPA,Internship_Experience,Projects,Certifications,AptitudeTestScore,Communication_Skills,Placement_Status,Technical_Skills
0,7.7,1,1,0,69,4.0,NotPlaced,2.592593
1,7.6,2,3,1,80,4.3,NotPlaced,7.777778
2,7.0,1,1,1,64,4.1,NotPlaced,3.703704
3,7.9,1,3,2,90,4.8,Placed,7.777778
4,6.7,0,2,1,71,3.7,NotPlaced,4.074074


In [7]:
df.columns

Index(['CGPA', 'Internship_Experience', 'Projects', 'Certifications',
       'AptitudeTestScore', 'Communication_Skills', 'Placement_Status',
       'Technical_Skills'],
      dtype='object')

In [8]:
df["Placement_Label"] = df["Placement_Status"].map({
    "Placed": 1,
    "NotPlaced": 0
})


In [9]:
df.head()

,CGPA,Internship_Experience,Projects,Certifications,AptitudeTestScore,Communication_Skills,Placement_Status,Technical_Skills,Placement_Label
0,7.7,1,1,0,69,4.0,NotPlaced,2.592593,0
1,7.6,2,3,1,80,4.3,NotPlaced,7.777778,0
2,7.0,1,1,1,64,4.1,NotPlaced,3.703704,0
3,7.9,1,3,2,90,4.8,Placed,7.777778,1
4,6.7,0,2,1,71,3.7,NotPlaced,4.074074,0


In [10]:
df.columns

Index(['CGPA', 'Internship_Experience', 'Projects', 'Certifications',
       'AptitudeTestScore', 'Communication_Skills', 'Placement_Status',
       'Technical_Skills', 'Placement_Label'],
      dtype='object')

In [11]:
df["Internship_Flag"] = df["Internship_Experience"].apply(
    lambda x: 1 if x > 0 else 0
)


In [12]:
df.head()

,CGPA,Internship_Experience,Projects,Certifications,AptitudeTestScore,Communication_Skills,Placement_Status,Technical_Skills,Placement_Label,Internship_Flag
0,7.7,1,1,0,69,4.0,NotPlaced,2.592593,0,1
1,7.6,2,3,1,80,4.3,NotPlaced,7.777778,0,1
2,7.0,1,1,1,64,4.1,NotPlaced,3.703704,0,1
3,7.9,1,3,2,90,4.8,Placed,7.777778,1,1
4,6.7,0,2,1,71,3.7,NotPlaced,4.074074,0,0


In [13]:
# Check missing values
df.isnull().sum()


CGPA                     0
Internship_Experience    0
Projects                 0
Certifications           0
AptitudeTestScore        0
Communication_Skills     0
Placement_Status         0
Technical_Skills         0
Placement_Label          0
Internship_Flag          0
dtype: int64

In [14]:
df.dtypes


CGPA                     float64
Internship_Experience      int64
Projects                   int64
Certifications             int64
AptitudeTestScore          int64
Communication_Skills     float64
Placement_Status          object
Technical_Skills         float64
Placement_Label            int64
Internship_Flag            int64
dtype: object

In [15]:
df.duplicated().sum()


957

In [16]:
df.drop_duplicates(inplace=True)


In [17]:
df.duplicated().sum()


0

In [18]:
df.shape


(6268, 10)

# detecting outliers

In [19]:
import numpy as np

numeric_cols = [
    "CGPA",
    "Internship_Experience",
    "Projects",
    "Certifications",
    "AptitudeTestScore",
    "Communication_Skills",
    "Technical_Skills"
]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}: {outliers.shape[0]} outliers")


CGPA: 0 outliers
Internship_Experience: 2579 outliers
Projects: 0 outliers
Certifications: 0 outliers
AptitudeTestScore: 0 outliers
Communication_Skills: 34 outliers
Technical_Skills: 0 outliers


In [20]:
Q1 = df["Communication_Skills"].quantile(0.25)
Q3 = df["Communication_Skills"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["Communication_Skills"] = df["Communication_Skills"].clip(
    lower_bound, upper_bound
)


In [21]:


numeric_cols = [
    "CGPA",
    "Internship_Experience",
    "Projects",
    "Certifications",
    "AptitudeTestScore",
    "Communication_Skills",
    "Technical_Skills"
]

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col}: {outliers.shape[0]} outliers")



CGPA: 0 outliers
Internship_Experience: 2579 outliers
Projects: 0 outliers
Certifications: 0 outliers
AptitudeTestScore: 0 outliers
Communication_Skills: 0 outliers
Technical_Skills: 0 outliers


In [23]:
df.head()

,CGPA,Internship_Experience,Projects,Certifications,AptitudeTestScore,Communication_Skills,Placement_Status,Technical_Skills,Placement_Label,Internship_Flag
0,7.7,1,1,0,69,4.0,NotPlaced,2.592593,0,1
1,7.6,2,3,1,80,4.3,NotPlaced,7.777778,0,1
2,7.0,1,1,1,64,4.1,NotPlaced,3.703704,0,1
3,7.9,1,3,2,90,4.8,Placed,7.777778,1,1
4,6.7,0,2,1,71,3.7,NotPlaced,4.074074,0,0


In [25]:
df.to_csv("preprocessed_data.csv", index=False)
